In [5]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [6]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [3]:
# Function to generate a new dataset2
import json

# prompt에 format 채점을 위해 새롭게 들어간 코드 존재 "format": * Focus 등

def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex"
    },
    ...additional
]
```
* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [4]:
# Generate the dataset2 and write it to 'dataset.json'
dataset2 = generate_dataset()
with open("dataset2.json", "w") as f:
    json.dump(dataset2, f, indent=2)

In [7]:
# 모델을 사용하여 테스트 케이스에 대한 클로드의 답변 을 평가하는 채점 로직
def grade_by_model(test_case, output):
    eval_prompt = f"""
당신은 숙련된 AWS 코드 리뷰어입니다. 아래 AI가 생성한 솔루션을 평가해주세요.

원본 과제:
<task>
{test_case["task"]}
</task>

평가 대상 솔루션:
<solution>
{output}
</solution>

출력 형식
다음 필드를 포함한 구조화된 JSON 객체로 평가 결과를 작성하세요. 필드 순서는 아래와 같이 유지합니다:
- "strengths": 핵심 강점 1~3개를 담은 배열
- "weaknesses": 개선이 필요한 영역 1~3개를 담은 배열
- "reasoning": 전반적인 평가에 대한 간결한 설명
- "score": 1~10 사이의 숫자

JSON 형식으로만 응답하세요. 답변은 간결하고 직접적으로 작성합니다.

응답 예시 구조:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """
    messages = []
    add_user_message(messages, eval_prompt)
    # haiku 모델은 여전히 message_prefilling을 지원하므로 강의 예시대로 이를 사용함
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [8]:
# format 검사를 위한 코드 기반 채점 함수들 / 학습자료 복붙
# Functions to validate the output structure
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

In [11]:
import json

# 클로드 개별 업무 지시용 함수
def run_prompt(test_case):
    """프롬프트와 테스트 케이스 입력값을 병합한 후 결과를 반환"""
    prompt = f"""
    다음 작업을 수행하라:
    
    * Respond only with Python, JSON, or a plain Regex
    * Do not add any comments or commentary or explanation

    {test_case["task"]}
    """

    
    messages = []
    add_user_message(messages, prompt)
    # 맥스 토큰 내로 점수를 계산하기 위한 간결화 system prompt
    # python json regex 명시하지 않고도 셋 중 하나의 형식으로 시작하게 만드는 message refilling
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output

# 클로드 개별 항목 검토 및 채점용 함수
def run_test_case(test_case):
    """run_prompt를 호출한 후 결과를 채점"""
    # 문제 먼저 풀고
    output = run_prompt(test_case)

    # 채점로직_ 그 답을 검토시키고
    model_grade= grade_by_model(test_case, output)
    # 검토 결과에서 점수만 추출
    model_score = model_grade["score"]
    # 검토 결과에서 설명만 추출
    reasoning = model_grade["reasoning"]

    # 코드 기반 채점에 관한 score 추가 후 모델 기반 채점 값과 평균(가중치를 동일하게 설정)하여 점수 산정
    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

# 파이썬에서 평균 계산기 호출

from statistics import mean

# 위 함수를 활용하여 수행한 작업 및 이에 대한 채점 결과를 순차적으로 제시
def run_eval(dataset):
    """데이터셋을 로드하고 각 테스트 케이스에 run_test_case를 호출"""
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    # 모든 테스트 케이스의 점수만 뽑아내어 이를 평균하여 제시
    
    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")
    return results

with open("dataset2.json", "r") as f:
    dataset = json.load(f)


In [12]:
results = run_eval(dataset)
# 결과값을 unicode가 아닌 한글로 인출하기 위해 ensure_ascii=False를 추가
json.dumps(results, indent=2, ensure_ascii=False)

Average score: 8.333333333333334


'[\n  {\n    "output": "\\nimport re\\nimport json\\n\\ndef parse_s3_uri(uri):\\n    match = re.match(r\'s3://([^/]+)/(.*)\', uri)\\n    if match:\\n        return {\\n            \'bucket\': match.group(1),\\n            \'key\': match.group(2)\\n        }\\n    return None\\n\\nuri = \'s3://my-bucket/path/to/object.txt\'\\nresult = parse_s3_uri(uri)\\nprint(json.dumps(result))\\n",\n    "test_case": {\n      "task": "Parse an AWS S3 bucket name and object key from a full S3 URI like \'s3://my-bucket/path/to/object.txt\' and extract the bucket name and key separately",\n      "format": "regex"\n    },\n    "score": 8.0,\n    "reasoning": "솔루션은 기본적인 S3 URI 파싱의 핵심 로직은 제대로 구현했으나, 프로덕션 환경에서 필요한 입력값 검증, Edge case 처리, 예외처리가 부족합니다. AWS 서비스와 연동하는 실무 코드로는 더 견고한 구현이 필요합니다."\n  },\n  {\n    "output": "\\n{\\n  \\"AWSTemplateFormatVersion\\": \\"2010-09-09\\",\\n  \\"Resources\\": {\\n    \\"BasicSecurityGroup\\": {\\n      \\"Type\\": \\"AWS::EC2::SecurityGroup\\",\\n      \\"Properties\\": {\\n